In [1]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np
import os
import glob
from scipy.signal import savgol_filter
import pandas as pd
import seaborn as sns
import math
from scipy.stats import exponnorm
import scipy.special as sse
from scipy.optimize import curve_fit
from scipy import stats
#from pomegranate import *
from sklearn.metrics import mean_squared_error
from sklearn.mixture import GaussianMixture
from scipy.stats import norm
from scipy.special import rel_entr
from sklearn.neighbors import KernelDensity

In [2]:
file_location = os.path.join('C:\\Users\\Austin\\Desktop\\cell_model\\fig_data', '*imt*')
filenames = glob.glob(file_location)

In [3]:
IMTs = []
for f in filenames:
    tmp_df = pd.read_csv(f)
    tmp_df = tmp_df.replace(to_replace=-1, value=np.nan)
    IMTs.append(tmp_df)

ii=0
for df in IMTs:
    df["IMT"] = df["Division"] - df["Mitotic Frame"]
    df["G1"] = df["Green On"] - df["Mitotic Frame"]
    df["S-M"] = df["Division"] - df["Green On"]
    if ii == 2 or ii == 3 or ii == 6 or ii == 7:
        df["IMT_t"] = df["IMT"] * (32/60)
        df["G1_t"] = df["G1"] * (32/60)
        df["S-M_t"] = df["S-M"] * (32/60)
    else:
        df["IMT_t"] = df["IMT"] * (20/60)
        df["G1_t"] = df["G1"] * (20/60)
        df["S-M_t"] = df["S-M"] * (20/60)
    #df = df[(df["IMT_t"] ==27)].reset_index(drop=True)
    ii+=1
    
stiffs = [0.2, 0.2, 0.6, 0.6, 4, 4, 55, 55]
ii=0
for df in IMTs:
    tmp_stiff = np.ones(len(df))*stiffs[ii]
    df.insert(0, "Stiffness", tmp_stiff)
    if ii % 2 == 0:
        df["Drug"] = "DMSO"
    else:
        df["Drug"] = "ERL"    
    ii+=1

all_IMTs = pd.concat(IMTs, ignore_index=True, sort=False)
centers = np.arange(0.5, 100.5, 1)

In [4]:
def calculate_aic(LL, num_params):
    aic = 2*num_params - 2*LL
    return aic

In [5]:
def calculate_aicc(AIC, num_params, samp_size):
    aic = AIC + (2*num_params*(num_params+1)/(samp_size-num_params-1))
    return aic

In [7]:
def calculate_bic(LL, num_params, samp_size):
    bic = num_params*math.log(samp_size) - 2*LL
    return bic

In [16]:
def calc_aicc(AIC, num_params, samp_size):
    aicc = AIC + (2*num_params*(num_params+1)/(samp_size-num_params-1))
    return aicc

In [25]:
def calc_aics(data):
    x = np.array(data)
    x = x[~np.isnan(x)]
    x_res = x.reshape(-1,1)

    # Fit models with 1-10 components
    k_arr = np.arange(3) + 1
    models = [GaussianMixture(n_components=k).fit(x_res) for k in k_arr]
    # Compute metrics to determine best hyperparameter
    AIC = [m.aic(x_res) for m in models]
    BIC = [m.bic(x_res) for m in models]
    
    params = exponnorm.fit(x)
    exp_c = 1/(params[0]*params[2])
    LLH = exponnorm.logpdf(x,*params).sum()
    emg_aic = calculate_aic(LLH, 3)
    emg_bic = calculate_bic(LLH, 3, len(x))
    AIC.append(emg_aic)
    BIC.append(emg_bic)
    
    aicc = []
    params = [2, 5, 8, 3]
    for n in range(4):
        tmp = calc_aicc(AIC[n], params[n], len(x))
        aicc.append(tmp)
        
    
    return AIC, BIC, aicc

In [30]:
def calc_aicp(AICs):
    aicp = [math.exp((min(AICs)-AIC)/2) for AIC in AICs]
    return aicp

In [53]:
[AICs, BICs, aicc] = calc_aics(IMTs[7]["G1_t"])

C:\Users\Austin\anaconda3\envs\stoch_cell\lib\site-packages\sklearn\cluster\_kmeans.py:1036: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [29]:
aicc

[2064.4973574498363, 1953.6610207268288, 1969.144611667766, 1977.4812680619116]

In [54]:
aicp = calc_aicp(aicc)

In [55]:
aicp

[2.547341386647959e-14, 0.015149877388487682, 0.004707211030447499, 1.0]

In [96]:
[AICs, BICs, aicc] = calc_aics(IMTs[7]["G1_t"])
aicp = calc_aicp(aicc)
aicp

C:\Users\Austin\anaconda3\envs\stoch_cell\lib\site-packages\sklearn\cluster\_kmeans.py:1036: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


[2.547341386647959e-14, 0.015270761498937382, 0.004707211030446964, 1.0]

In [71]:
[AICs, BICs, aicc] = calc_aics(IMTs[5]["IMT_t"])
aicp = calc_aicp(aicc)
aicp

C:\Users\Austin\anaconda3\envs\stoch_cell\lib\site-packages\sklearn\cluster\_kmeans.py:1036: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


[4.425260675804177e-12, 1.0, 0.002461902319227724, 0.16060401757692108]